# 2 · CNN baseline

Trains the TextCNN that every pretrained model has to beat. Runs in minutes,
on CPU if necessary.

**Covers the assignment's CNN implementation requirement.**


## Setup

Clone the repository and install. The data layer needs nothing beyond the standard
library, so this is only for the model code.


In [ ]:
!git clone -q https://github.com/ManasDasri/NNDL.git
%cd NNDL
!pip install -q -e . 'matplotlib>=3.8'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — set Runtime > Change runtime type > T4 GPU')


### Prepare the corpus

Extracts answer spans from `CUAD_v1.json` and assigns document-level splits.
Chunking happens at training time, because each model needs a different window.


In [ ]:
!legal-risk-prepare --cuad_json data/CUAD_v1.json --out_dir data/processed


### Train

Loss is BCE weighted by each label's negative-to-positive ratio. Without it the
model converges on predicting nothing, which the dataset report explains.


In [ ]:
!legal-risk-train-cnn --epochs 12 --batch_size 32 --lr 1e-3 --output_dir outputs/cnn


### Evaluate at both levels

Window level asks whether a clause is recognised in the window in front of the model.
Document level asks whether the contract gets flagged at all. They are different
questions and both are reported.


In [ ]:
!legal-risk-evaluate --run_dir outputs/cnn --split test --pooling max


### Why max-pooling

Compare the three rules. The task is existential: a clause in one window of a
sixty-window contract means the contract contains it.


In [ ]:
for method in ('max', 'mean', 'top_k'):
    !legal-risk-evaluate --run_dir outputs/cnn --split test --pooling {method} --out /tmp/eval_{method}.json


### Ablation: what the weighted loss is doing

Train the same model without `pos_weight` and watch it collapse toward the
negative class. This is the clearest single demonstration of the imbalance.


In [ ]:
!legal-risk-train-cnn --epochs 6 --no_pos_weight --output_dir outputs/cnn_unweighted

import json
for run in ('outputs/cnn', 'outputs/cnn_unweighted'):
    m = json.load(open(f'{run}/metrics.json'))['test_at_tuned_thresholds']
    print(f"{run:28s} macro F1 {m['macro_f1']:.3f}")
    for row in m['per_class']:
        print(f"    {row['label']:30s} F1 {row['f1']:.3f}  predicted {row['predicted']:4d} / {row['support']:3d} actual")
